# 沪深300成分股回测工作流示例

本示例展示如何使用增强版缓存数据服务进行沪深300成分股回测，解决新股上市导致的数据缺失问题。

In [ ]:
# 导入必要的模块
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd

# VNPY相关模块
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import HistoryRequest, BarData
from vnpy.trader.datafeed import get_datafeed
from vnpy.trader.cached_datafeed import (
    EnhancedCachedDatafeedWrapper,
    create_enhanced_cached_datafeed,
    StockMetadataManager,
    DataStatus
)
from tools.hs300_backtest_helper import HS300BacktestHelper

## 1. 初始化回测辅助工具

In [ ]:
# 创建回测辅助工具
helper = HS300BacktestHelper(cache_root="./data_cache")

# 获取沪深300成分股列表
stocks = helper.get_hs300_constituents()
print(f"沪深300成分股数量: {len(stocks)}")
print(f"前10只成分股: {stocks[:10]}")

## 2. 设置回测参数

In [ ]:
# 回测时间窗口
BACKTEST_START = datetime(2020, 1, 1)
BACKTEST_END = datetime(2024, 12, 31)

# 数据周期
DATA_INTERVAL = Interval.DAILY

# 最小数据要求（天）
MIN_DATA_DAYS = 30

print(f"回测时间窗口: {BACKTEST_START} ~ {BACKTEST_END}")
print(f"数据周期: {DATA_INTERVAL.value}")

## 3. 筛选有效股票

根据回测时间窗口筛选有效股票，排除：
- 回测开始后才上市的股票
- 回测开始前已退市的股票
- 数据不足的股票

In [ ]:
# 获取有效股票列表
valid_stocks = helper.get_valid_stocks_for_backtest(
    start_date=BACKTEST_START,
    end_date=BACKTEST_END,
    min_data_days=MIN_DATA_DAYS
)

# 统计结果
valid_count = sum(1 for s in valid_stocks if s['is_valid'])
invalid_count = len(valid_stocks) - valid_count

print(f"\n有效股票: {valid_count} 只")
print(f"无效股票: {invalid_count} 只")

# 分析无效原因
invalid_reasons = {}
for stock in valid_stocks:
    if not stock['is_valid']:
        reason = stock['reason']
        invalid_reasons[reason] = invalid_reasons.get(reason, 0) + 1

print("\n无效原因分布:")
for reason, count in invalid_reasons.items():
    print(f"  {reason}: {count} 只 ({count/invalid_count*100:.1f}%)")

## 4. 批量下载数据

使用增强版数据服务批量下载数据，自动处理：
- 跳过已缓存的完整数据
- 记录上市/退市时间
- 标记获取失败的股票

In [ ]:
# 筛选出有效的股票代码
selected_stocks = [s['vt_symbol'] for s in valid_stocks if s['is_valid']]
print(f"\n准备下载 {len(selected_stocks)} 只股票的数据...")

# 批量下载数据（先测试前10只）
test_stocks = selected_stocks[:10]  # 生产环境去掉此限制

result = helper.download_stock_data_batch(
    vt_symbols=test_stocks,
    start_date=BACKTEST_START,
    end_date=BACKTEST_END,
    interval=DATA_INTERVAL,
    skip_existing=True  # 跳过已完整缓存的股票
)

# 打印下载结果
print(f"\n下载结果:")
print(f"  总数: {result['total']}")
print(f"  已下载: {result['downloaded']}")
print(f"  已跳过: {result['skipped']}")
print(f"  失败: {result['failed']}")

# 显示详细结果
print("\n详细结果:")
for detail in result['details']:
    status = detail['status']
    symbol = detail['vt_symbol']
    if status == 'downloaded':
        print(f"  ✓ {symbol}: 下载 {detail['count']} 条数据")
    elif status == 'skipped':
        print(f"  - {symbol}: 跳过 ({detail['reason']})")
    else:
        print(f"  ✗ {symbol}: 失败 ({detail['reason']})")

## 5. 查看数据状态报告

In [ ]:
# 获取数据状态报告
report = helper.get_data_status_report()
print(report)

## 6. 使用增强版数据服务查询数据

演示如何使用增强版数据服务进行数据查询，自动处理上市/退市时间检查。

In [ ]:
# 获取增强版数据服务
datafeed = helper.cached_datafeed

# 查询单只股票数据
symbol = "000001"
exchange = Exchange.SZSE

# 创建请求（故意设置在上市前的日期）
req = HistoryRequest(
    symbol=symbol,
    exchange=exchange,
    interval=DATA_INTERVAL,
    start=datetime(2000, 1, 1),  # 远在上市之前
    end=datetime(2024, 1, 1)
)

# 查询数据（会自动调整时间范围）
bars = datafeed.query_bar_history(req, print)

print(f"\n查询结果: {len(bars)} 条数据")
if bars:
    print(f"数据范围: {bars[0].datetime} ~ {bars[-1].datetime}")
    print(f"\n第一条数据:")
    print(f"  日期: {bars[0].datetime}")
    print(f"  开盘: {bars[0].open_price}")
    print(f"  最高: {bars[0].high_price}")
    print(f"  最低: {bars[0].low_price}")
    print(f"  收盘: {bars[0].close_price}")
    print(f"  成交量: {bars[0].volume}")

## 7. 验证上市时间自动检测

展示系统如何自动检测股票上市时间。

In [ ]:
# 获取股票元信息
metadata = helper.metadata_manager.get_metadata(symbol, exchange)

if metadata:
    print(f"\n股票 {symbol}.{exchange.value} 元信息:")
    print(f"  上市日期: {metadata.listed_date}")
    print(f"  退市日期: {metadata.delisted_date or '未退市'}")
    print(f"  实际首笔交易: {metadata.first_trade_date}")
    print(f"  实际末笔交易: {metadata.last_trade_date}")
    print(f"  数据状态: {metadata.data_status.value}")
    print(f"  获取尝试次数: {metadata.fetch_attempts}")
    print(f"  Parquet缓存范围: {metadata.parquet_start} ~ {metadata.parquet_end}")
    print(f"  数据库缓存范围: {metadata.db_start} ~ {metadata.db_end}")
else:
    print(f"未找到 {symbol}.{exchange.value} 的元信息")

## 8. 一致性检查

验证数据库和Parquet缓存的一致性。

In [ ]:
# 验证缓存一致性
consistency = helper.metadata_manager.validate_cache_consistency(
    symbol=symbol,
    exchange=exchange,
    interval=DATA_INTERVAL
)

print(f"\n一致性检查结果:")
print(f"  一致: {consistency['consistent']}")

if consistency['issues']:
    print("  问题列表:")
    for issue in consistency['issues']:
        print(f"    - {issue}")

if consistency['db_overview']:
    db = consistency['db_overview']
    print(f"  数据库缓存: {db.count} 条, {db.start} ~ {db.end}")

if consistency['parquet_overview']:
    pq = consistency['parquet_overview']
    print(f"  Parquet缓存: {pq['start']} ~ {pq['end']}")

## 9. 在回测引擎中使用

展示如何将增强版数据服务集成到回测引擎中。

In [ ]:
# 示例：在回测前过滤股票
def filter_stocks_for_backtest(
    vt_symbols: list,
    start_date: datetime,
    end_date: datetime,
    metadata_manager: StockMetadataManager
    ) -> list:
    """
    过滤出回测时间窗口内有效的股票
    """
    valid_symbols = []
    
    for vt_symbol in vt_symbols:
        # 解析股票代码
        parts = vt_symbol.split(".")
        symbol = parts[0]
        exchange = Exchange(parts[1])
        
        # 检查日期范围
        result = metadata_manager.check_date_range(
            symbol, exchange, start_date, end_date
        )
        
        if result['valid']:
            valid_symbols.append(vt_symbol)
            
            # 如果需要调整时间范围，可以在这里处理
            if result['message']:
                print(f"  {vt_symbol}: {result['message']}")
        else:
            print(f"  {vt_symbol}: 跳过 ({result['message']})")
    
    return valid_symbols

# 使用示例
filtered_stocks = filter_stocks_for_backtest(
    vt_symbols=test_stocks,
    start_date=BACKTEST_START,
    end_date=BACKTEST_END,
    metadata_manager=helper.metadata_manager
)

print(f"\n过滤后有效股票: {len(filtered_stocks)} 只")
print(f"过滤后股票列表: {filtered_stocks}")

## 总结

通过使用增强版缓存数据服务，可以：

1. **自动检测上市/退市时间**：在首次获取数据时自动记录
2. **避免重复获取不存在的数据**：对于未上市或已退市的股票，跳过网络请求
3. **智能调整查询时间范围**：自动将查询范围限制在上市/退市时间内
4. **记录数据获取状态**：标记获取失败的股票，避免重复尝试
5. **一致性检查**：验证数据库和Parquet缓存的一致性
6. **批量数据管理**：支持批量下载和状态管理

这些功能特别适合沪深300成分股回测场景，解决了成分股调整和新股上市带来的数据缺失问题。